# L08 · 멀티턴 agent OPD

## Goal

**예상 시간:** 35분 · **경로:** full

- turn과 환경 state를 표현한다
- 오류 누적을 관찰한다
- single-turn 가정을 벗어난다

### 현재 위치: L07 → **L08** → L09

```text
Prompt/Data -> state source -> ... -> L08 -> ... -> fair evaluation
```

Alt text: The course map highlights L08 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L08"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L08', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

멀티턴에서는 action이 환경 observation을 바꾼다. 초반 한 번의 오류가 이후 teacher에게 낯선 state를 만들 수 있으므로 token loss만 보고 성공을 판단할 수 없다.

그림 대체 설명: 출력의 label과 숫자는 색 없이도 읽을 수 있다.

### 핵심 원리

멀티턴에서는 trajectory가 단순 text가 아니라 `(observation_t, action_t, next observation, terminal)`의 연쇄다. student action이 transition을 바꾸므로 teacher가 원래 정답 trajectory에서만 잘해도 student state에서는 회복하지 못할 수 있다.

turn/step boundary를 token별로 보존하면 어떤 step이 실패했는지, environment token을 loss에서 제외했는지, terminal 뒤 token을 잘못 학습했는지 감사할 수 있다. sequence-level 성공과 token-level teacher agreement를 함께 기록해야 한다.

### 실제 구현: 왜 이렇게 만들었나

환경 state는 immutable record이고 `step(action)`이 다음 observation과 terminal/success를 반환한다. multi-turn batch는 prompt token의 turn ID를 `-1`, response turn을 `0..N-1`로 둬 mask slicing을 안전하게 한다.

실제 코드: [`calculator.py`](../../src/opd_study/envs/calculator.py), [`tokenizer.py`](../../src/opd_study/data/tokenizer.py).

In [2]:
import inspect
from opd_study.envs import CalculatorEnvironment

objects_to_show = (CalculatorEnvironment.step,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.envs.calculator.CalculatorEnvironment.step
    def step(self, proposed_value: int) -> CalculatorState:
        if self._terminal:
            raise RuntimeError("cannot step a terminal environment; call reset")
        operator, operand = self._operations[self._turn]
        correct_value = self._apply(self._value, operator, operand)
        correct = proposed_value == correct_value
        # The environment uses the student's actual value. One wrong turn therefore
        # changes every later state and makes error compounding visible.
        self._value = proposed_value
        self._turn += 1
        self._terminal = self._turn == len(self._operations)
        success = self._terminal and self._value == self._target
        if self._terminal:
            observation = (
                f"Done; value {self._value}; target {self._target}; "
                f"{'success' if success else 'failure'}"
            )
        else:
            next_operator, next_operand = self.

### 다른 선택지는 없나?

환경을 text transcript로만 저장할 수도 있지만 observation/action 구조를 잃는다. 반대로 완전한 simulator snapshot은 정확하지만 크다. 최소한 turn ID, terminal, environment observation hash와 action을 보존하는 절충이 필요하다.

### 2/3 · 실행하고 관찰하기

실행 전 예측: L08의 첫 출력에서 가장 먼저 확인해야 할 invariant는 무엇일까? 한 문장으로 적고 실행한다.

In [3]:
from opd_study.envs import CalculatorEnvironment

environment = CalculatorEnvironment(2, (("+", 3), ("*", 4)))
print(environment.reset().observation)
after_error = environment.step(6)
print(after_error.observation)
final = environment.step(24)
print(final.observation)

Current 2; apply + 3
Tool says expected 5; current 6; apply * 4
Done; value 24; target 20; failure


In [4]:
environment.reset()
correct_first = environment.step(5)
correct_final = environment.step(20)
print("correct path:", correct_first.observation, "->", correct_final.observation)
print("The next observation depends on the student's previous action.")

correct path: Tool says correct; current 5; apply * 4 -> Done; value 20; target 20; success
The next observation depends on the student's previous action.


## Checks

In [5]:
assert after_error.value == 6
assert final.target == 20 and not final.success
assert correct_final.success
print("check passed: an early action changes later states and final success")

check passed: an early action changes later states and final success


**연습 (8분):** 같은 최종 숫자라도 첫 action이 다른 두 trajectory를 만들고 observation trace가 같은지 비교하라.

<details><summary>확인 기준</summary>environment transition 때문에 중간 state가 다르면 token sequence만 같은지와 별개로 trajectory provenance가 다르다.</details>

## 내가 자주 틀리는 것

### M1 — 멀티턴을 긴 single-turn text로만 보기

- 틀린 형태: turn/observation 경계를 버린다.
- 왜 틀렸나: action이 다음 state를 바꾸는 인과를 감사할 수 없다.
- 고친 형태: turn IDs, terminal과 observation을 보존한다.
- 관련 검사: `test_an_early_error_changes_later_state`

### M2 — token agreement를 task success로 부르기

- 틀린 형태: teacher token과 많이 같으면 환경 성공이라고 한다.
- 왜 틀렸나: 한 핵심 action 오류가 전체 task를 실패시킬 수 있다.
- 고친 형태: sequence success와 token 지표를 함께 기록한다.
- 관련 검사: `test_an_early_error_changes_later_state`

## 60초 요약

1. turn과 환경 state를 표현한다
2. 오류 누적을 관찰한다
3. single-turn 가정을 벗어난다

## Next Steps

다음 노트북으로 가기 전, 위 assertion을 다시 실행하고 틀린 예측 한 줄을 남긴다.

### Sources

- [`tcod`](https://arxiv.org/abs/2604.24005v3) · `2604.24005v3` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)
- [`sod`](https://arxiv.org/abs/2605.07725v3) · `2605.07725v3` · license `arXiv-non-exclusive-distribution-1.0` · [audited manifest](../../docs/sources.yml)
- [`sage_opd`](https://arxiv.org/abs/2606.19659v1) · `2606.19659v1` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)